# Employee Salary Hike Prediction
- **Problem Type**: Regression
- **Dataset**: Kaggle – HR Analytics / Indian Tech Salaries Dataset (22,770 records)
- **Objective**: Predict salary increment percentage (`Percentage Salary Hike`)
- **Input Features**: Experience, Performance Rating, Role (plus Salary, Location, Status)
- **ML Techniques**: Linear Regression (Baseline), Gradient Boosting Regressor (Ensemble)
- **Metrics**: Root Mean Squared Error (RMSE), R² Score, Mean Absolute Error (MAE)
- **Outcome**: HR Decision Support Tool

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Load Dataset
df = pd.read_csv('salary_data.csv')
print(f'Total records: {df.shape[0]}, Total columns: {df.shape[1]}')
df.head()

## 1. Exploratory Data Analysis & Target Distribution

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isnull().sum())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df['Percentage Salary Hike'], bins=25, color='royalblue', edgecolor='black')
plt.title('Distribution of Salary Hike (%)')
plt.xlabel('Percentage Salary Hike (%)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
df.groupby('Job Roles')['Percentage Salary Hike'].mean().plot(kind='bar', color='teal', edgecolor='black')
plt.title('Mean Salary Hike by Job Role')
plt.xlabel('Job Role')
plt.ylabel('Mean Hike (%)')
plt.tight_layout()
plt.show()

## 2. Feature Preprocessing & Model Training Pipelines

In [ ]:
target = 'Percentage Salary Hike'
y = df[target]

# Experiment 1: Core Required Features (Experience, Rating, Role)
num_core = ['Experience', 'Rating']
cat_core = ['Job Roles']
X_core = df[num_core + cat_core]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_core, y, test_size=0.2, random_state=42)

preprocessor_core = ColumnTransformer([
    ('num', StandardScaler(), num_core),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_core)
])

# Pipeline A: Linear Regression
pipe_lr = Pipeline([
    ('prep', preprocessor_core),
    ('model', LinearRegression())
])
pipe_lr.fit(X_train_c, y_train_c)
y_pred_lr = pipe_lr.predict(X_test_c)

# Pipeline B: Gradient Boosting Regressor
pipe_gb = Pipeline([
    ('prep', preprocessor_core),
    ('model', GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42))
])
pipe_gb.fit(X_train_c, y_train_c)
y_pred_gb = pipe_gb.predict(X_test_c)

print('Core Models Trained Successfully!')

## 3. Evaluation Metrics Comparison (RMSE, R², MAE)

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {'Model': name, 'RMSE': round(rmse, 4), 'R2 Score': round(r2, 4), 'MAE (%)': round(mae, 4)}

metrics = pd.DataFrame([
    evaluate('Linear Regression (Core)', y_test_c, y_pred_lr),
    evaluate('Gradient Boosting (Core)', y_test_c, y_pred_gb)
])
metrics

## 4. HR Decision Support Inference Function

In [ ]:
def recommend_salary_hike(experience_years, performance_rating, job_role, current_salary):
    inp = pd.DataFrame([{
        'Experience': experience_years,
        'Rating': performance_rating,
        'Job Roles': job_role
    }])
    hike_pred = pipe_gb.predict(inp)[0]
    hike_pred = max(5.0, min(35.0, hike_pred))
    new_sal = current_salary * (1 + hike_pred / 100.0)
    
    print(f'=== HR Decision Recommendation ===')
    print(f'Role               : {job_role}')
    print(f'Experience         : {experience_years} years')
    print(f'Performance Rating : {performance_rating} / 5.0')
    print(f'Current Base Salary: INR {current_salary:,.2f}')
    print(f'Recommended Hike   : {hike_pred:.2f}%')
    print(f'Adjusted New Salary: INR {new_sal:,.2f}')
    return hike_pred, new_sal

# Example Execution
recommend_salary_hike(experience_years=5, performance_rating=4.2, job_role='SDE', current_salary=600000);